# 99 Clear Test Data

Notebook n?y d?ng ?? d?n d? li?u test tr??c khi ch?y lu?ng th?t.

M?c ??nh `dry_run=true`, ch? in ra d? li?u s? x?a. ??i `confirm_delete=true` v? `dry_run=false` m?i x?a th?t.

Notebook x?a d? li?u trong S3 prefix v? drop b?ng Delta trong catalog. Bucket v?n gi? nguy?n.

In [ ]:
# Safety widgets
try:
    dbutils.widgets.text("s3_bucket", "de-e2e-413612133697-ap-southeast-1-an")
    dbutils.widgets.text("catalog", "de_e2e")
    dbutils.widgets.text("dry_run", "true")
    dbutils.widgets.text("confirm_delete", "false")
except NameError:
    raise RuntimeError("Run this notebook on Databricks")

bucket = dbutils.widgets.get("s3_bucket").strip()
catalog = dbutils.widgets.get("catalog").strip()
dry_run = dbutils.widgets.get("dry_run").strip().lower() == "true"
confirm_delete = dbutils.widgets.get("confirm_delete").strip().lower() == "true"

if not bucket:
    raise ValueError("s3_bucket is required")
if not catalog:
    raise ValueError("catalog is required")

print(f"bucket={bucket}")
print(f"catalog={catalog}")
print(f"dry_run={dry_run}")
print(f"confirm_delete={confirm_delete}")

## 1. ??nh ngh?a v?ng c?n x?a

Ch? x?a c?c prefix thu?c project n?y:
- `lakehouse/landing/douyin/`
- `lakehouse/bronze/douyin/`
- `lakehouse/silver/douyin/`
- `lakehouse/gold/douyin/`
- `lakehouse/control/douyin/`

Kh?ng x?a bucket, kh?ng x?a prefix ngo?i `lakehouse/.../douyin`.

In [ ]:
s3_prefixes = [
    f"s3://{bucket}/lakehouse/landing/douyin/",
    f"s3://{bucket}/lakehouse/bronze/douyin/",
    f"s3://{bucket}/lakehouse/silver/douyin/",
    f"s3://{bucket}/lakehouse/gold/douyin/",
    f"s3://{bucket}/lakehouse/control/douyin/",
]

schemas = ["bronze", "silver", "gold"]

known_tables = [
    f"{catalog}.bronze.douyin_aweme_raw",
    f"{catalog}.bronze.douyin_media_manifest_raw",
    f"{catalog}.silver.douyin_aweme_clean",
    f"{catalog}.silver.douyin_aweme_hashtag",
    f"{catalog}.silver.douyin_aweme_media",
    f"{catalog}.silver.aweme_snapshot_delta",
    f"{catalog}.gold.dim_date",
    f"{catalog}.gold.dim_creator",
    f"{catalog}.gold.dim_aweme",
    f"{catalog}.gold.dim_hashtag",
    f"{catalog}.gold.dim_media",
    f"{catalog}.gold.fact_aweme_daily_performance",
    f"{catalog}.gold.fact_creator_daily_performance",
    f"{catalog}.gold.fact_hashtag_daily",
]

print("S3 prefixes to clear:")
for path in s3_prefixes:
    print(f"- {path}")

print("\nKnown tables to drop if exists:")
for table in known_tables:
    print(f"- {table}")

## 2. Ki?m tra d? li?u hi?n c?

In [ ]:
def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

print("S3 prefix status:")
for path in s3_prefixes:
    exists = path_exists(path)
    print(f"- {path}: {'exists' if exists else 'missing'}")

print("\nTable status:")
for table in known_tables:
    exists = spark.catalog.tableExists(table)
    print(f"- {table}: {'exists' if exists else 'missing'}")

## 3. Drop b?ng Delta trong catalog

In [ ]:
for table in known_tables:
    sql = f"DROP TABLE IF EXISTS {table}"
    if dry_run:
        print(f"DRY RUN: {sql}")
    else:
        if not confirm_delete:
            raise ValueError("Set confirm_delete=true before deleting data")
        print(sql)
        spark.sql(sql)

## 4. X?a file S3 theo prefix

In [ ]:
def safe_remove_prefix(path: str) -> None:
    allowed_prefix = f"s3://{bucket}/lakehouse/"
    if not path.startswith(allowed_prefix):
        raise ValueError(f"Unsafe path outside project lakehouse prefix: {path}")
    if not path.rstrip("/").endswith("douyin") and "/douyin/" not in path:
        raise ValueError(f"Unsafe path must target douyin data: {path}")
    dbutils.fs.rm(path, recurse=True)

for path in s3_prefixes:
    if dry_run:
        print(f"DRY RUN: dbutils.fs.rm({path}, recurse=True)")
    else:
        if not confirm_delete:
            raise ValueError("Set confirm_delete=true before deleting data")
        print(f"Removing {path}")
        safe_remove_prefix(path)

## 5. T?o l?i schema tr?ng

In [ ]:
for schema in schemas:
    sql = f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}"
    if dry_run:
        print(f"DRY RUN: {sql}")
    else:
        print(sql)
        spark.sql(sql)

print("Done. Ready for real CSV upload and Airflow run.")